# EquiNexus public detector/refiner inference

This run applies the public Apache-2.0 I079 detector/refiner checkpoints to the 180 supplied competition test images. The Kaggle model card says the weights were trained only on the official competition training images and annotations, and reports no test-overlap ground truth. Since the shared checkpoint was trained on all competition training observations, it is excluded from our grouped holdout validation; this is a test-inference candidate only.

EquiNexus independently implements the model architecture and inference adapter in the embedded source bundle. We do not copy the publisher's notebook source. The notebook does not read training annotations or test labels, and it does not submit automatically. It validates the CSV and writes it under `/kaggle/working/equinexus/public-i079/` for review.

Model card: <https://www.kaggle.com/models/phuongncn/solar-filament-i079-weights>  
Published inference-method reference: <https://www.kaggle.com/code/robinamirbahar/solar-filament-panoptic-morphology-segmentation>  
Source commit: `d9e23b697796c600acd6b05d0a12362888388976`  
Embedded-source SHA-256: `2a90331207116ee758f0b315e4be30910c488ac0cbe791343abb176055d223a6`


In [ ]:
import base64
from pathlib import Path
REPOSITORY_PATH = Path('/kaggle/working/EquiNexus')
SOURCE_COMMIT = 'd9e23b697796c600acd6b05d0a12362888388976'
SOURCE_BUNDLE_SHA256 = '2a90331207116ee758f0b315e4be30910c488ac0cbe791343abb176055d223a6'
SOURCE_FILES = {"src/filament_baseline/data.py":"IyBDb3B5cmlnaHQgKEMpIDIwMjYgRXF1aU5leHVzIGNvbnRyaWJ1dG9ycwojIFNQRFgtTGljZW5zZS1JZGVudGlmaWVyOiBBR1BMLTMuMC1vbmx5Cgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQganNvbgppbXBvcnQgcmUKZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgZGVmYXVsdGRpY3QKZnJvbSBkYXRhY2xhc3NlcyBpbXBvcnQgZGF0YWNsYXNzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgpBTk5PVEFUSU9OX0ZJTEVOQU1FID0gIk1BR0ZpTE9fMS4wX0Fubm90YXRpb25zX2thZ2dsZTIwMjZfdHJhaW4uanNvbiIKSU1BR0VfU0laRSA9IDIwNDgKCgpAZGF0YWNsYXNzKGZyb3plbj1UcnVlKQpjbGFzcyBJbWFnZVZpZXc6CiAgICBpbWFnZV9pZDogc3RyCiAgICBmaWxlX25hbWU6IHN0cgogICAgd2lkdGg6IGludAogICAgaGVpZ2h0OiBpbnQKICAgIHllYXI6IGludAogICAgc2l0ZTogc3RyCgoKQGRhdGFjbGFzcyhmcm96ZW49VHJ1ZSkKY2xhc3MgQ29tcGV0aXRpb25EYXRhc2V0OgogICAgcm9vdDogUGF0aAogICAgdHJhaW5faW1hZ2VfZGlyOiBQYXRoCiAgICB0ZXN0X2ltYWdlX2RpcjogUGF0aAogICAgYW5ub3RhdGlvbl9wYXRoOiBQYXRoCiAgICBpbWFnZV92aWV3czogdHVwbGVbSW1hZ2VWaWV3LCAuLi5dCiAgICBhbm5vdGF0aW9uc19ieV9pbWFnZV9pZDogZGljdFtzdHIsIHR1cGxlW2RpY3Rbc3RyLCBBbnldLCAuLi5dXQogICAgdmlld3NfYnlfb2JzZXJ2YXRpb246IGRpY3Rbc3RyLCB0dXBsZVtJbWFnZVZpZXcsIC4uLl1dCiAgICBjYXRlZ29yaWVzOiB0dXBsZVtkaWN0W3N0ciwgQW55XSwgLi4uXQogICAgaW1hZ2VfcmVzaXplX3RvOiB0dXBsZVtpbnQsIGludF0gfCBOb25lID0gTm9uZQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIG9ic2VydmF0aW9uX25hbWVzKHNlbGYpIC0+IHR1cGxlW3N0ciwgLi4uXToKICAgICAgICByZXR1cm4gdHVwbGUoc29ydGVkKHNlbGYudmlld3NfYnlfb2JzZXJ2YXRpb24pKQoKICAgIEBwcm9wZXJ0eQogICAgZGVmIHRlc3RfaW1hZ2VfY291bnQoc2VsZikgLT4gaW50OgogICAgICAgIHJldHVybiBzdW0ocGF0aC5pc19maWxlKCkgZm9yIHBhdGggaW4gc2VsZi50ZXN0X2ltYWdlX2Rpci5pdGVyZGlyKCkpCgogICAgQHByb3BlcnR5CiAgICBkZWYgYW5ub3RhdGlvbl9jb3VudChzZWxmKSAtPiBpbnQ6CiAgICAgICAgcmV0dXJuIHN1bShsZW4oaXRlbXMpIGZvciBpdGVtcyBpbiBzZWxmLmFubm90YXRpb25zX2J5X2ltYWdlX2lkLnZhbHVlcygpKQoKCmRlZiBfbG9va3NfbGlrZV9kYXRhc2V0KHJvb3Q6IFBhdGgpIC0+IGJvb2w6CiAgICByZXR1cm4gKAogICAgICAgIChyb290IC8gInRyYWluIiAvIEFOTk9UQVRJT05fRklMRU5BTUUpLmlzX2ZpbGUoKQogICAgICAgIGFuZCAocm9vdCAvICJ0cmFpbiIgLyAidHJhaW5faW1hZ2VzIikuaXNfZGlyKCkKICAgICAgICBhbmQgKHJvb3QgLyAidGVzdCIgLyAidGVzdF9pbWFnZXMiKS5pc19kaXIoKQogICAgKQoKCmRlZiBmaW5kX2RhdGFzZXRfcm9vdChzZWFyY2hfcm9vdDogc3RyIHwgUGF0aCkgLT4gUGF0aDoKICAgICIiIkZpbmQgdGhlIGNvbXBldGl0aW9uIGlucHV0IGZvbGRlciB3aXRob3V0IHJlbHlpbmcgb24gYSBmaXhlZCBLYWdnbGUgbW91bnQgbmFtZS4iIiIKICAgIHNlYXJjaF9yb290ID0gUGF0aChzZWFyY2hfcm9vdCkuZXhwYW5kdXNlcigpLnJlc29sdmUoKQogICAgaWYgbm90IHNlYXJjaF9yb290LmV4aXN0cygpOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiRGF0YSBzZWFyY2ggcGF0aCBkb2VzIG5vdCBleGlzdDoge3NlYXJjaF9yb290fSIpCiAgICBpZiBfbG9va3NfbGlrZV9kYXRhc2V0KHNlYXJjaF9yb290KToKICAgICAgICByZXR1cm4gc2VhcmNoX3Jvb3QKCiAgICBtYXRjaGVzID0gc29ydGVkKAogICAgICAgIHtwYXRoLnBhcmVudC5wYXJlbnQucmVzb2x2ZSgpIGZvciBwYXRoIGluIHNlYXJjaF9yb290LnJnbG9iKEFOTk9UQVRJT05fRklMRU5BTUUpfQogICAgICAgIGlmIHNlYXJjaF9yb290LmlzX2RpcigpCiAgICAgICAgZWxzZSBzZXQoKQogICAgKQogICAgbWF0Y2hlcyA9IFtwYXRoIGZvciBwYXRoIGluIG1hdGNoZXMgaWYgX2xvb2tzX2xpa2VfZGF0YXNldChwYXRoKV0KICAgIGlmIGxlbihtYXRjaGVzKSA9PSAxOgogICAgICAgIHJldHVybiBtYXRjaGVzWzBdCiAgICBpZiBub3QgbWF0Y2hlczoKICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcigKICAgICAgICAgICAgZiJDb3VsZCBub3QgZmluZCB7QU5OT1RBVElPTl9GSUxFTkFNRX0gd2l0aCB0cmFpbi90cmFpbl9pbWFnZXMgYW5kICIKICAgICAgICAgICAgZiJ0ZXN0L3Rlc3RfaW1hZ2VzIGJlbG93IHtzZWFyY2hfcm9vdH0iCiAgICAgICAgKQogICAgcmVuZGVyZWQgPSAiXG4iLmpvaW4oc3RyKHBhdGgpIGZvciBwYXRoIGluIG1hdGNoZXMpCiAgICByYWlzZSBSdW50aW1lRXJyb3IoCiAgICAgICAgIk1vcmUgdGhhbiBvbmUgY29tcGV0aXRpb24gZGF0YXNldCB3YXMgZm91bmQ7IHBhc3MgYW4gZXhwbGljaXQgZGF0YXNldCByb290OlxuIgogICAgICAgIGYie3JlbmRlcmVkfSIKICAgICkKCgpkZWYgX3llYXJfYW5kX3NpdGUoZmlsZV9uYW1lOiBzdHIpIC0+IHR1cGxlW2ludCwgc3RyXToKICAgIHN0ZW0gPSBQYXRoKGZpbGVfbmFtZSkuc3RlbQogICAgaWYgbGVuKHN0ZW0pIDwgNCBvciBub3Qgc3RlbVs6NF0uaXNkaWdpdCgpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJDYW5ub3QgZGVyaXZlIGNhcHR1cmUgeWVhciBmcm9tIGltYWdlIGZpbGVuYW1lOiB7ZmlsZV9uYW1lfSIpCiAgICB5ZWFyID0gaW50KHN0ZW1bOjRdKQogICAgc2l0ZV9tYXRjaCA9IHJlLmZ1bGxtYXRjaChyIlxkezE0fShbQS1aYS16XXsyfSkiLCBzdGVtKQogICAgc2l0ZSA9IHNpdGVfbWF0Y2guZ3JvdXAoMSkgaWYgc2l0ZV9tYXRjaCBlbHNlICJ1bmtub3duIgogICAgcmV0dXJuIHllYXIsIHNpdGUKCgpkZWYgbG9hZF9kYXRhc2V0KGRhdGFfcm9vdDogc3RyIHwgUGF0aCkgLT4gQ29tcGV0aXRpb25EYXRhc2V0OgogICAgcm9vdCA9IGZpbmRfZGF0YXNldF9yb290KGRhdGFfcm9vdCkKICAgIGFubm90YXRpb25fcGF0aCA9IHJvb3QgLyAidHJhaW4iIC8gQU5OT1RBVElPTl9GSUxFTkFNRQogICAgdHJhaW5faW1hZ2VfZGlyID0gcm9vdCAvICJ0cmFpbiIgLyAidHJhaW5faW1hZ2VzIgogICAgdGVzdF9pbWFnZV9kaXIgPSByb290IC8gInRlc3QiIC8gInRlc3RfaW1hZ2VzIgogICAgcGF5bG9hZCA9IGpzb24ubG9hZHMoYW5ub3RhdGlvbl9wYXRoLnJlYWRfdGV4dChlbmNvZGluZz0idXRmLTgiKSkKCiAgICBpbWFnZV92aWV3czogbGlzdFtJbWFnZVZpZXddID0gW10KICAgIGltYWdlX2lkczogc2V0W3N0cl0gPSBzZXQoKQogICAgdmlld3NfYnlfb2JzZXJ2YXRpb246IGRpY3Rbc3RyLCBsaXN0W0ltYWdlVmlld11dID0gZGVmYXVsdGRpY3QobGlzdCkKICAgIGZvciByZWNvcmQgaW4gcGF5bG9hZC5nZXQoImltYWdlcyIsIFtdKToKICAgICAgICBpbWFnZV9pZCA9IHN0cihyZWNvcmRbImlkIl0pCiAgICAgICAgZmlsZV9uYW1lID0gc3RyKHJlY29yZFsiZmlsZV9uYW1lIl0pCiAgICAgICAgaWYgUGF0aChmaWxlX25hbWUpLm5hbWUgIT0gZmlsZV9uYW1lOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiRXhwZWN0ZWQgYSBiYXNlbmFtZSBpbiBDT0NPIGZpbGVfbmFtZSwgZ290OiB7ZmlsZV9uYW1lfSIpCiAgICAgICAgaWYgaW1hZ2VfaWQgaW4gaW1hZ2VfaWRzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiRHVwbGljYXRlIENPQ08gaW1hZ2UgaWQ6IHtpbWFnZV9pZH0iKQogICAgICAgIGltYWdlX2lkcy5hZGQoaW1hZ2VfaWQpCiAgICAgICAgeWVhciwgc2l0ZSA9IF95ZWFyX2FuZF9zaXRlKGZpbGVfbmFtZSkKICAgICAgICB2aWV3ID0gSW1hZ2VWaWV3KAogICAgICAgICAgICBpbWFnZV9pZD1pbWFnZV9pZCwKICAgICAgICAgICAgZmlsZV9uYW1lPWZpbGVfbmFtZSwKICAgICAgICAgICAgd2lkdGg9aW50KHJlY29yZFsid2lkdGgiXSksCiAgICAgICAgICAgIGhlaWdodD1pbnQocmVjb3JkWyJoZWlnaHQiXSksCiAgICAgICAgICAgIHllYXI9eWVhciwKICAgICAgICAgICAgc2l0ZT1zaXRlLAogICAgICAgICkKICAgICAgICBpZiB2aWV3LndpZHRoICE9IElNQUdFX1NJWkUgb3Igdmlldy5oZWlnaHQgIT0gSU1BR0VfU0laRToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigKICAgICAgICAgICAgICAgIGYiRXhwZWN0ZWQge0lNQUdFX1NJWkV9eHtJTUFHRV9TSVpFfSwgZ290IHt2aWV3LndpZHRofXh7dmlldy5oZWlnaHR9ICIKICAgICAgICAgICAgICAgIGYiZm9yIHtmaWxlX25hbWV9IgogICAgICAgICAgICApCiAgICAgICAgaW1hZ2Vfdmlld3MuYXBwZW5kKHZpZXcpCiAgICAgICAgdmlld3NfYnlfb2JzZXJ2YXRpb25bZmlsZV9uYW1lXS5hcHBlbmQodmlldykKCiAgICBhbm5vdGF0aW9uczogZGljdFtzdHIsIGxpc3RbZGljdFtzdHIsIEFueV1dXSA9IGRlZmF1bHRkaWN0KGxpc3QpCiAgICBmb3IgYW5ub3RhdGlvbiBpbiBwYXlsb2FkLmdldCgiYW5ub3RhdGlvbnMiLCBbXSk6CiAgICAgICAgaW1hZ2VfaWQgPSBzdHIoYW5ub3RhdGlvblsiaW1hZ2VfaWQiXSkKICAgICAgICBpZiBpbWFnZV9pZCBub3QgaW4gaW1hZ2VfaWRzOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiQW5ub3RhdGlvbiByZWZlcmVuY2VzIHVua25vd24gaW1hZ2UgaWQ6IHtpbWFnZV9pZH0iKQogICAgICAgIGlmIGludChhbm5vdGF0aW9uLmdldCgiaXNjcm93ZCIsIDApKSAhPSAwOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJUaGUgY29tcGV0aXRpb24gdHJhaW5pbmcgYW5ub3RhdGlvbnMgdW5leHBlY3RlZGx5IGNvbnRhaW4gY3Jvd2QgbWFza3MiKQogICAgICAgIGFubm90YXRpb25zW2ltYWdlX2lkXS5hcHBlbmQoYW5ub3RhdGlvbikKCiAgICBmb3IgaW1hZ2VfaWQgaW4gaW1hZ2VfaWRzOgogICAgICAgIGFubm90YXRpb25zW2ltYWdlX2lkXS5zb3J0KGtleT1sYW1iZGEgaXRlbTogc3RyKGl0ZW1bImlkIl0pKQogICAgICAgIGlmIG5vdCBhbm5vdGF0aW9uc1tpbWFnZV9pZF06CiAgICAgICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJBbm5vdGF0b3IgdmlldyB7aW1hZ2VfaWR9IGhhcyBubyBmaWxhbWVudCBhbm5vdGF0aW9ucyIpCgogICAgZm9yIGZpbGVfbmFtZSwgdmlld3MgaW4gdmlld3NfYnlfb2JzZXJ2YXRpb24uaXRlbXMoKToKICAgICAgICBleHBlY3RlZF9wYXRoID0gdHJhaW5faW1hZ2VfZGlyIC8gZmlsZV9uYW1lCiAgICAgICAgaWYgbm90IGV4cGVjdGVkX3BhdGguaXNfZmlsZSgpOgogICAgICAgICAgICByYWlzZSBGaWxlTm90Rm91bmRFcnJvcihmIkNPQ08gaW1hZ2UgZmlsZSBpcyBtaXNzaW5nOiB7ZXhwZWN0ZWRfcGF0aH0iKQogICAgICAgIHZpZXdzLnNvcnQoa2V5PWxhbWJkYSB2aWV3OiB2aWV3LmltYWdlX2lkKQoKICAgIHRlc3RfZmlsZXMgPSBbcGF0aCBmb3IgcGF0aCBpbiB0ZXN0X2ltYWdlX2Rpci5pdGVyZGlyKCkgaWYgcGF0aC5pc19maWxlKCldCiAgICBpZiBub3QgdGVzdF9maWxlczoKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYiTm8gdGVzdCBpbWFnZXMgZm91bmQgaW4ge3Rlc3RfaW1hZ2VfZGlyfSIpCgogICAgcmV0dXJuIENvbXBldGl0aW9uRGF0YXNldCgKICAgICAgICByb290PXJvb3QsCiAgICAgICAgdHJhaW5faW1hZ2VfZGlyPXRyYWluX2ltYWdlX2RpciwKICAgICAgICB0ZXN0X2ltYWdlX2Rpcj10ZXN0X2ltYWdlX2RpciwKICAgICAgICBhbm5vdGF0aW9uX3BhdGg9YW5ub3RhdGlvbl9wYXRoLAogICAgICAgIGltYWdlX3ZpZXdzPXR1cGxlKHNvcnRlZChpbWFnZV92aWV3cywga2V5PWxhbWJkYSB2aWV3OiAodmlldy5maWxlX25hbWUsIHZpZXcuaW1hZ2VfaWQpKSksCiAgICAgICAgYW5ub3RhdGlvbnNfYnlfaW1hZ2VfaWQ9ewogICAgICAgICAgICBpbWFnZV9pZDogdHVwbGUoaXRlbXMpIGZvciBpbWFnZV9pZCwgaXRlbXMgaW4gYW5ub3RhdGlvbnMuaXRlbXMoKQogICAgICAgIH0sCiAgICAgICAgdmlld3NfYnlfb2JzZXJ2YXRpb249ewogICAgICAgICAgICBmaWxlX25hbWU6IHR1cGxlKHZpZXdzKQogICAgICAgICAgICBmb3IgZmlsZV9uYW1lLCB2aWV3cyBpbiBzb3J0ZWQodmlld3NfYnlfb2JzZXJ2YXRpb24uaXRlbXMoKSkKICAgICAgICB9LAogICAgICAgIGNhdGVnb3JpZXM9dHVwbGUocGF5bG9hZC5nZXQoImNhdGVnb3JpZXMiLCBbXSkpLAogICAgKQoKCmRlZiBpbnZlbnRvcnkoZGF0YXNldDogQ29tcGV0aXRpb25EYXRhc2V0KSAtPiBkaWN0W3N0ciwgaW50XToKICAgIHJldHVybiB7CiAgICAgICAgIm9ic2VydmF0aW9uX2NvdW50IjogbGVuKGRhdGFzZXQudmlld3NfYnlfb2JzZXJ2YXRpb24pLAogICAgICAgICJhbm5vdGF0b3Jfdmlld19jb3VudCI6IGxlbihkYXRhc2V0LmltYWdlX3ZpZXdzKSwKICAgICAgICAiYW5ub3RhdGlvbl9jb3VudCI6IGRhdGFzZXQuYW5ub3RhdGlvbl9jb3VudCwKICAgICAgICAidHJhaW5faW1hZ2VfZmlsZV9jb3VudCI6IHN1bSgKICAgICAgICAgICAgcGF0aC5pc19maWxlKCkgZm9yIHBhdGggaW4gZGF0YXNldC50cmFpbl9pbWFnZV9kaXIuaXRlcmRpcigpCiAgICAgICAgKSwKICAgICAgICAidGVzdF9pbWFnZV9maWxlX2NvdW50IjogZGF0YXNldC50ZXN0X2ltYWdlX2NvdW50LAogICAgfQo=","src/filament_baseline/runtime.py":"IyBDb3B5cmlnaHQgKEMpIDIwMjYgRXF1aU5leHVzIGNvbnRyaWJ1dG9ycwojIFNQRFgtTGljZW5zZS1JZGVudGlmaWVyOiBBR1BMLTMuMC1vbmx5Cgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgaW1wb3J0bGliLm1ldGFkYXRhCmltcG9ydCBwbGF0Zm9ybQppbXBvcnQgc3VicHJvY2VzcwppbXBvcnQgc3lzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApmcm9tIHR5cGluZyBpbXBvcnQgQW55CgoKVFJBQ0tFRF9SVU5USU1FX1BBQ0tBR0VTID0gKAogICAgIm51bXB5IiwKICAgICJvcGVuY3YtcHl0aG9uLWhlYWRsZXNzIiwKICAgICJwYW5kYXMiLAogICAgInNjaWtpdC1sZWFybiIsCiAgICAic2NpcHkiLAogICAgIm1hdHBsb3RsaWIiLAogICAgInB5Y29jb3Rvb2xzIiwKICAgICJ0b3JjaCIsCikKCgpkZWYgcGFja2FnZV92ZXJzaW9ucygpIC0+IGRpY3Rbc3RyLCBzdHIgfCBOb25lXToKICAgIHZlcnNpb25zOiBkaWN0W3N0ciwgc3RyIHwgTm9uZV0gPSB7fQogICAgZm9yIHBhY2thZ2UgaW4gVFJBQ0tFRF9SVU5USU1FX1BBQ0tBR0VTOgogICAgICAgIHRyeToKICAgICAgICAgICAgdmVyc2lvbnNbcGFja2FnZV0gPSBpbXBvcnRsaWIubWV0YWRhdGEudmVyc2lvbihwYWNrYWdlKQogICAgICAgIGV4Y2VwdCBpbXBvcnRsaWIubWV0YWRhdGEuUGFja2FnZU5vdEZvdW5kRXJyb3I6CiAgICAgICAgICAgIHZlcnNpb25zW3BhY2thZ2VdID0gTm9uZQogICAgcmV0dXJuIHZlcnNpb25zCgoKZGVmIGhhcmR3YXJlX2luZm8oKSAtPiBkaWN0W3N0ciwgQW55XToKICAgIHJlc3VsdDogZGljdFtzdHIsIEFueV0gPSB7CiAgICAgICAgInBsYXRmb3JtIjogcGxhdGZvcm0ucGxhdGZvcm0oKSwKICAgICAgICAicHl0aG9uIjogc3lzLnZlcnNpb24uc3BsaXQoKVswXSwKICAgICAgICAiY3VkYV9hdmFpbGFibGUiOiBGYWxzZSwKICAgICAgICAidG9yY2hfdmVyc2lvbiI6IE5vbmUsCiAgICAgICAgInRvcmNoX2N1ZGFfdmVyc2lvbiI6IE5vbmUsCiAgICAgICAgImdwdV9kZXZpY2VzIjogW10sCiAgICAgICAgIm52aWRpYV9zbWlfc3VtbWFyeSI6IE5vbmUsCiAgICB9CiAgICB0cnk6CiAgICAgICAgcmVzdWx0WyJudmlkaWFfc21pX3N1bW1hcnkiXSA9IHN1YnByb2Nlc3MucnVuKAogICAgICAgICAgICBbCiAgICAgICAgICAgICAgICAibnZpZGlhLXNtaSIsCiAgICAgICAgICAgICAgICAiLS1xdWVyeS1ncHU9bmFtZSxtZW1vcnkudG90YWwsZHJpdmVyX3ZlcnNpb24iLAogICAgICAgICAgICAgICAgIi0tZm9ybWF0PWNzdixub2hlYWRlciIsCiAgICAgICAgICAgIF0sCiAgICAgICAgICAgIGNoZWNrPVRydWUsCiAgICAgICAgICAgIGNhcHR1cmVfb3V0cHV0PVRydWUsCiAgICAgICAgICAgIHRleHQ9VHJ1ZSwKICAgICAgICApLnN0ZG91dC5zdHJpcCgpLnNwbGl0bGluZXMoKQogICAgZXhjZXB0IChPU0Vycm9yLCBzdWJwcm9jZXNzLkNhbGxlZFByb2Nlc3NFcnJvcik6CiAgICAgICAgcGFzcwogICAgdHJ5OgogICAgICAgIGltcG9ydCB0b3JjaAogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIHJldHVybiByZXN1bHQKCiAgICByZXN1bHRbInRvcmNoX3ZlcnNpb24iXSA9IHRvcmNoLl9fdmVyc2lvbl9fCiAgICByZXN1bHRbInRvcmNoX2N1ZGFfdmVyc2lvbiJdID0gdG9yY2gudmVyc2lvbi5jdWRhCiAgICByZXN1bHRbImN1ZGFfYXZhaWxhYmxlIl0gPSBib29sKHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkpCiAgICBpZiByZXN1bHRbImN1ZGFfYXZhaWxhYmxlIl06CiAgICAgICAgZGV2aWNlcyA9IFtdCiAgICAgICAgZm9yIGluZGV4IGluIHJhbmdlKHRvcmNoLmN1ZGEuZGV2aWNlX2NvdW50KCkpOgogICAgICAgICAgICBwcm9wcyA9IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9wcm9wZXJ0aWVzKGluZGV4KQogICAgICAgICAgICBkZXZpY2VzLmFwcGVuZCgKICAgICAgICAgICAgICAgIHsKICAgICAgICAgICAgICAgICAgICAiaW5kZXgiOiBpbmRleCwKICAgICAgICAgICAgICAgICAgICAibmFtZSI6IHByb3BzLm5hbWUsCiAgICAgICAgICAgICAgICAgICAgInRvdGFsX21lbW9yeV9ieXRlcyI6IGludChwcm9wcy50b3RhbF9tZW1vcnkpLAogICAgICAgICAgICAgICAgICAgICJjb21wdXRlX2NhcGFiaWxpdHkiOiBmIntwcm9wcy5tYWpvcn0ue3Byb3BzLm1pbm9yfSIsCiAgICAgICAgICAgICAgICB9CiAgICAgICAgICAgICkKICAgICAgICByZXN1bHRbImdwdV9kZXZpY2VzIl0gPSBkZXZpY2VzCiAgICAgICAgdG9yY2guY3VkYS5yZXNldF9wZWFrX21lbW9yeV9zdGF0cygpCiAgICByZXR1cm4gcmVzdWx0CgoKZGVmIGdpdF9pbmZvKHJlcG9fcm9vdDogc3RyIHwgUGF0aCkgLT4gZGljdFtzdHIsIEFueV06CiAgICByZXBvX3Jvb3QgPSBQYXRoKHJlcG9fcm9vdCkKICAgIHRyeToKICAgICAgICBjb21taXQgPSBzdWJwcm9jZXNzLnJ1bigKICAgICAgICAgICAgWyJnaXQiLCAiLUMiLCBzdHIocmVwb19yb290KSwgInJldi1wYXJzZSIsICJIRUFEIl0sCiAgICAgICAgICAgIGNoZWNrPVRydWUsCiAgICAgICAgICAgIGNhcHR1cmVfb3V0cHV0PVRydWUsCiAgICAgICAgICAgIHRleHQ9VHJ1ZSwKICAgICAgICApLnN0ZG91dC5zdHJpcCgpCiAgICAgICAgc3RhdHVzID0gc3VicHJvY2Vzcy5ydW4oCiAgICAgICAgICAgIFsiZ2l0IiwgIi1DIiwgc3RyKHJlcG9fcm9vdCksICJzdGF0dXMiLCAiLS1wb3JjZWxhaW4iXSwKICAgICAgICAgICAgY2hlY2s9VHJ1ZSwKICAgICAgICAgICAgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwKICAgICAgICAgICAgdGV4dD1UcnVlLAogICAgICAgICkuc3Rkb3V0CiAgICAgICAgcmV0dXJuIHsiY29tbWl0IjogY29tbWl0LCAid29ya3RyZWVfZGlydHkiOiBib29sKHN0YXR1cy5zdHJpcCgpKX0KICAgIGV4Y2VwdCAoT1NFcnJvciwgc3VicHJvY2Vzcy5DYWxsZWRQcm9jZXNzRXJyb3IpOgogICAgICAgIHJldHVybiB7ImNvbW1pdCI6IE5vbmUsICJ3b3JrdHJlZV9kaXJ0eSI6IE5vbmV9Cg==","src/filament_baseline/external_panoptic.py":"...","scripts/run_external_panoptic_inference.py":"IyBDb3B5cmlnaHQgKEMpIDIwMjYgRXF1aU5leHVzIGNvbnRyaWJ1dG9ycwojIFNQRFgtTGljZW5zZS1JZGVudGlmaWVyOiBBR1BMLTMuMC1vbmx5Cgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGNzdgppbXBvcnQgaGFzaGxpYgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHBsYXRmb3JtCmltcG9ydCB0aW1lCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aAoKaW1wb3J0IGN2MgppbXBvcnQgcGFuZGFzIGFzIHBkCmltcG9ydCB0b3JjaApmcm9tIHB5Y29jb3Rvb2xzIGltcG9ydCBtYXNrIGFzIG1hc2tfdXRpbHMKCmZyb20gZmlsYW1lbnRfYmFzZWxpbmUuZGF0YSBpbXBvcnQgZmluZF9kYXRhc2V0X3Jvb3QKZnJvbSBmaWxhbWVudF9iYXNlbGluZS5leHRlcm5hbF9wYW5vcHRpYyBpbXBvcnQgKAogICAgRVhQRUNURURfTU9ERUxfTUQ1LAogICAgZW5jb2RlX2luc3RhbmNlLAogICAgbG9hZF9wdWJsaWNfY2hlY2twb2ludHMsCiAgICBwcmVkaWN0X2ltYWdlLAopCmZyb20gZmlsYW1lbnRfYmFzZWxpbmUucnVudGltZSBpbXBvcnQgaGFyZHdhcmVfaW5mbywgcGFja2FnZV92ZXJzaW9ucwoKClJFUE9fUk9PVCA9IFBhdGgoX19maWxlX18pLnJlc29sdmUoKS5wYXJlbnRzWzFdCk1PREVMX1NPVVJDRSA9ICJwaHVvbmduY24vc29sYXItZmlsYW1lbnQtaTA3OS13ZWlnaHRzL1B5VG9yY2gvZGV0ZWN0b3ItcmVmaW5lci8xIgoKCmRlZiBfZGlnZXN0KHBhdGg6IFBhdGgsIGFsZ29yaXRobTogc3RyKSAtPiBzdHI6CiAgICBkaWdlc3QgPSBoYXNobGliLm5ldyhhbGdvcml0aG0pCiAgICB3aXRoIHBhdGgub3BlbigicmIiKSBhcyBzdHJlYW06CiAgICAgICAgZm9yIGJsb2NrIGluIGl0ZXIobGFtYmRhOiBzdHJlYW0ucmVhZCgxMDI0ICogMTAyNCksIGIiIik6CiAgICAgICAgICAgIGRpZ2VzdC51cGRhdGUoYmxvY2spCiAgICByZXR1cm4gZGlnZXN0LmhleGRpZ2VzdCgpCgoKZGVmIF9sb2NhdGVfY2hlY2twb2ludChzZWFyY2hfcm9vdDogUGF0aCwgbmFtZTogc3RyLCBtZDU6IHN0cikgLT4gUGF0aDoKICAgIG1hdGNoZXMgPSBbCiAgICAgICAgcGF0aCBmb3IgcGF0aCBpbiBzZWFyY2hfcm9vdC5yZ2xvYihuYW1lKQogICAgICAgIGlmIHBhdGguaXNfZmlsZSgpIGFuZCBfZGlnZXN0KHBhdGgsICJtZDUiKSA9PSBtZDUKICAgIF0KICAgIGlmIGxlbihtYXRjaGVzKSAhPSAxOgogICAgICAgIHJhaXNlIEZpbGVOb3RGb3VuZEVycm9yKGYiRXhwZWN0ZWQgb25lIG1kNS12ZXJpZmllZCB7bmFtZX0gYmVsb3cge3NlYXJjaF9yb290fSwgZm91bmQge2xlbihtYXRjaGVzKX0iKQogICAgcmV0dXJuIG1hdGNoZXNbMF0KCgpkZWYgX3ZhbGlkYXRlX3N1Ym1pc3Npb24oZnJhbWU6IHBkLkRhdGFGcmFtZSwgdGVzdF9zdGVtczogc2V0W3N0cl0pIC0+IGRpY3Rbc3RyLCBpbnRdOgogICAgaWYgbGlzdChmcmFtZS5jb2x1bW5zKSAhPSBbImZpbGFtZW50X2lkIiwgInNlZ21lbnRhdGlvbl9ybGUiXToKICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJzdWJtaXNzaW9uIGNvbHVtbnMgZGlmZmVyIGZyb20gdGhlIG9yZ2FuaXplciBmb3JtYXQiKQogICAgaWYgZnJhbWUuZW1wdHkgb3IgZnJhbWUuaXNuYSgpLmFueSgpLmFueSgpIG9yIG5vdCBmcmFtZS5maWxhbWVudF9pZC5pc191bmlxdWU6CiAgICAgICAgcmFpc2UgVmFsdWVFcnJvcigic3VibWlzc2lvbiBtdXN0IGNvbnRhaW4gbm9uLWVtcHR5LCB1bmlxdWUsIGNvbXBsZXRlIHJvd3MiKQogICAgY292ZXJlZDogc2V0W3N0cl0gPSBzZXQoKQogICAgZm9yIHJvdyBpbiBmcmFtZS5pdGVydHVwbGVzKGluZGV4PUZhbHNlKToKICAgICAgICBpbWFnZV9zdGVtID0gc3RyKHJvdy5maWxhbWVudF9pZCkucnNwbGl0KCJfIiwgbWF4c3BsaXQ9MSlbMF0KICAgICAgICBpZiBpbWFnZV9zdGVtIG5vdCBpbiB0ZXN0X3N0ZW1zOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKGYic3VibWlzc2lvbiByZWZlcmVuY2VzIHVua25vd24gaW1hZ2Uge2ltYWdlX3N0ZW19IikKICAgICAgICBjb3ZlcmVkLmFkZChpbWFnZV9zdGVtKQogICAgICAgIGVuY29kZWQgPSBzdHIocm93LnNlZ21lbnRhdGlvbl9ybGUpLmVuY29kZSgiYXNjaWkiKQogICAgICAgIGRlY29kZWQgPSBtYXNrX3V0aWxzLmRlY29kZSh7InNpemUiOiBbMjA0OCwgMjA0OF0sICJjb3VudHMiOiBlbmNvZGVkfSkKICAgICAgICBpZiBkZWNvZGVkLnNoYXBlICE9ICgyMDQ4LCAyMDQ4KSBvciBub3QgZGVjb2RlZC5hbnkoKToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmImludmFsaWQgb3IgZW1wdHkgUkxFIGZvciB7cm93LmZpbGFtZW50X2lkfSIpCiAgICByZXR1cm4gewogICAgICAgICJyb3dzIjogaW50KGxlbihmcmFtZSkpLAogICAgICAgICJwcm9jZXNzZWRfaW1hZ2VzIjogaW50KGxlbih0ZXN0X3N0ZW1zKSksCiAgICAgICAgImltYWdlc193aXRoX3ByZWRpY3Rpb25zIjogaW50KGxlbihjb3ZlcmVkKSksCiAgICAgICAgImltYWdlc193aXRob3V0X3ByZWRpY3Rpb25zIjogaW50KGxlbih0ZXN0X3N0ZW1zIC0gY292ZXJlZCkpLAogICAgfQoKCmRlZiBtYWluKCkgLT4gaW50OgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoCiAgICAgICAgZGVzY3JpcHRpb249IlJ1biB0aGUgcHVibGljIEFwYWNoZS0yLjAgZGV0ZWN0b3IvcmVmaW5lciBjaGVja3BvaW50cyBvbiBjb21wZXRpdGlvbiB0ZXN0IGltYWdlcyBvbmx5LiIKICAgICkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGF0YS1yb290IiwgdHlwZT1QYXRoLCBkZWZhdWx0PVBhdGgoIi9rYWdnbGUvaW5wdXQiKSkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbW9kZWwtcm9vdCIsIHR5cGU9UGF0aCwgZGVmYXVsdD1QYXRoKCIva2FnZ2xlL2lucHV0IikpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW91dHB1dC1kaXIiLCB0eXBlPVBhdGgsIGRlZmF1bHQ9UGF0aCgiL2thZ2dsZS93b3JraW5nL2VxdWluZXh1cy9wdWJsaWMtaTA3OSIpKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1leHBlY3RlZC10ZXN0LWltYWdlcyIsIHR5cGU9aW50LCBkZWZhdWx0PTE4MCkKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCgogICAgZGF0YV9yb290ID0gYXJncy5kYXRhX3Jvb3QuZXhwYW5kdXNlcigpLnJlc29sdmUoKQogICAgbW9kZWxfcm9vdCA9IGFyZ3MubW9kZWxfcm9vdC5leHBhbmR1c2VyKCkucmVzb2x2ZSgpCiAgICBvdXRwdXRfZGlyID0gYXJncy5vdXRwdXRfZGlyLmV4cGFuZHVzZXIoKS5yZXNvbHZlKCkKICAgIGthZ2dsZV9ydW50aW1lID0gUGF0aCgiL2thZ2dsZS9pbnB1dCIpLmlzX2RpcigpCiAgICBhbGxvd2VkX291dHB1dF9yb290ID0gUGF0aCgiL2thZ2dsZS93b3JraW5nIikucmVzb2x2ZSgpIGlmIGthZ2dsZV9ydW50aW1lIGVsc2UgKFJFUE9fUk9PVCAvICJhcnRpZmFjdHMiKS5yZXNvbHZlKCkKICAgIGlmIG5vdCBvdXRwdXRfZGlyLmlzX3JlbGF0aXZlX3RvKGFsbG93ZWRfb3V0cHV0X3Jvb3QpOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJvdXRwdXRzIG11c3QgcmVtYWluIHVuZGVyIHthbGxvd2VkX291dHB1dF9yb290fSIpCiAgICBvdXRwdXRfZGlyLm1rZGlyKHBhcmVudHM9VHJ1ZSwgZXhpc3Rfb2s9VHJ1ZSkKCiAgICBjb21wZXRpdGlvbl9yb290ID0gZmluZF9kYXRhc2V0X3Jvb3QoZGF0YV9yb290KQogICAgdGVzdF9kaXIgPSBjb21wZXRpdGlvbl9yb290IC8gInRlc3QiIC8gInRlc3RfaW1hZ2VzIgogICAgdGVzdF9maWxlcyA9IHNvcnRlZChwYXRoIGZvciBwYXRoIGluIHRlc3RfZGlyLml0ZXJkaXIoKSBpZiBwYXRoLmlzX2ZpbGUoKSkKICAgIGlmIGxlbih0ZXN0X2ZpbGVzKSAhPSBhcmdzLmV4cGVjdGVkX3Rlc3RfaW1hZ2VzOgogICAgICAgIHJhaXNlIFZhbHVlRXJyb3IoZiJleHBlY3RlZCB7YXJncy5leHBlY3RlZF90ZXN0X2ltYWdlc30gdGVzdCBpbWFnZXMsIGZvdW5kIHtsZW4odGVzdF9maWxlcyl9IikKICAgIGRldGVjdG9yX3BhdGggPSBfbG9jYXRlX2NoZWNrcG9pbnQoCiAgICAgICAgbW9kZWxfcm9vdCwgImZpbmFsX2RldGVjdG9yLnB0IiwgRVhQRUNURURfTU9ERUxfTUQ1WyJmaW5hbF9kZXRlY3Rvci5wdCJdCiAgICApCiAgICByZWZpbmVyX3BhdGggPSBfbG9jYXRlX2NoZWNrcG9pbnQoCiAgICAgICAgbW9kZWxfcm9vdCwgImZpbmFsX3JlZmluZXIucHQiLCBFWFBFQ1RFRF9NT0RFTF9NRDVbImZpbmFsX3JlZmluZXIucHQiXQogICAgKQoKICAgIGlmIG5vdCB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgIHJhaXNlIFJ1bnRpbWVFcnJvcigidGhpcyBwdWJsaWMgY2hlY2twb2ludCBwYWlyIHJlcXVpcmVzIGEgQ1VEQSBkZXZpY2UiKQogICAgdG9yY2guY3VkYS5yZXNldF9wZWFrX21lbW9yeV9zdGF0cygpCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGE6MCIpCiAgICBkZXRlY3RvciwgcmVmaW5lciA9IGxvYWRfcHVibGljX2NoZWNrcG9pbnRzKGRldGVjdG9yX3BhdGgsIHJlZmluZXJfcGF0aCwgZGV2aWNlKQoKICAgIHJvd3M6IGxpc3RbZGljdFtzdHIsIHN0cl1dID0gW10KICAgIGRpYWdub3N0aWNfcm93cyA9IFtdCiAgICBzdGFydGVkID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgZm9yIGluZGV4LCBpbWFnZV9wYXRoIGluIGVudW1lcmF0ZSh0ZXN0X2ZpbGVzLCBzdGFydD0xKToKICAgICAgICBpbWFnZSA9IGN2Mi5pbXJlYWQoc3RyKGltYWdlX3BhdGgpLCBjdjIuSU1SRUFEX0dSQVlTQ0FMRSkKICAgICAgICBpZiBpbWFnZSBpcyBOb25lIG9yIGltYWdlLnNoYXBlICE9ICgyMDQ4LCAyMDQ4KToKICAgICAgICAgICAgcmFpc2UgVmFsdWVFcnJvcihmInRlc3QgaW1hZ2UgZmFpbGVkIGZ1bGwtcmVzb2x1dGlvbiBkZWNvZGU6IHtpbWFnZV9wYXRoLm5hbWV9IikKICAgICAgICBtYXNrcywgZGlhZ25vc3RpY3MgPSBwcmVkaWN0X2ltYWdlKGRldGVjdG9yLCByZWZpbmVyLCBpbWFnZSwgZGV2aWNlKQogICAgICAgIGltYWdlX3N0ZW0gPSBpbWFnZV9wYXRoLnN0ZW0KICAgICAgICBmb3IgaW5zdGFuY2VfaW5kZXgsIG1hc2sgaW4gZW51bWVyYXRlKG1hc2tzLCBzdGFydD0xKToKICAgICAgICAgICAgcm93cy5hcHBlbmQoewogICAgICAgICAgICAgICAgImZpbGFtZW50X2lkIjogZiJ7aW1hZ2Vfc3RlbX1fe2luc3RhbmNlX2luZGV4fSIsCiAgICAgICAgICAgICAgICAic2VnbWVudGF0aW9uX3JsZSI6IGVuY29kZV9pbnN0YW5jZShtYXNrKSwKICAgICAgICAgICAgfSkKICAgICAgICBkaWFnbm9zdGljX3Jvd3MuYXBwZW5kKHsiaW1hZ2UiOiBpbWFnZV9wYXRoLm5hbWUsICoqZGlhZ25vc3RpY3MsICJwcmVkaWN0aW9ucyI6IGxlbihtYXNrcyl9KQogICAgICAgIGlmIGluZGV4ICUgMTAgPT0gMCBvciBpbmRleCA9PSBsZW4odGVzdF9maWxlcyk6CiAgICAgICAgICAgIHByaW50KGpzb24uZHVtcHMoewogICAgICAgICAgICAgICAgImV2ZW50IjogImluZmVyZW5jZV9wcm9ncmVzcyIsCiAgICAgICAgICAgICAgICAiY29tcGxldGVkIjogaW5kZXgsCiAgICAgICAgICAgICAgICAidG90YWwiOiBsZW4odGVzdF9maWxlcyksCiAgICAgICAgICAgICAgICAiZWxhcHNlZF9zZWNvbmRzIjogcm91bmQodGltZS5wZXJmX2NvdW50ZXIoKSAtIHN0YXJ0ZWQsIDIpLAogICAgICAgICAgICB9KSwgZmx1c2g9VHJ1ZSkKCiAgICBmcmFtZSA9IHBkLkRhdGFGcmFtZShyb3dzLCBjb2x1bW5zPVsiZmlsYW1lbnRfaWQiLCAic2VnbWVudGF0aW9uX3JsZSJdKQogICAgY292ZXJhZ2UgPSBfdmFsaWRhdGVfc3VibWlzc2lvbihmcmFtZSwge3BhdGguc3RlbSBmb3IgcGF0aCBpbiB0ZXN0X2ZpbGVzfSkKICAgIHN1Ym1pc3Npb25fcGF0aCA9IG91dHB1dF9kaXIgLyAic3VibWlzc2lvbi5jc3YiCiAgICBmcmFtZS50b19jc3Yoc3VibWlzc2lvbl9wYXRoLCBpbmRleD1GYWxzZSwgcXVvdGluZz1jc3YuUVVPVEVfTUlOSU1BTCkKICAgIGVsYXBzZWQgPSB0aW1lLnBlcmZfY291bnRlcigpIC0gc3RhcnRlZAogICAgcmVwb3J0ID0gewogICAgICAgICJydW5fbmFtZSI6ICJwdWJsaWMtaTA3OS1hcGFjaGUyLXRlc3QtaW5mZXJlbmNlIiwKICAgICAgICAiY2FuZGlkYXRlX2lkIjogInB1YmxpYy1pMDc5LWRldGVjdG9yLXJlZmluZXItZmxpcDQtdHRhIiwKICAgICAgICAic291cmNlX21vZGVsIjogTU9ERUxfU09VUkNFLAogICAgICAgICJzb3VyY2VfbW9kZWxfbGljZW5zZSI6ICJBcGFjaGUtMi4wIiwKICAgICAgICAic291cmNlX21vZGVsX2NhcmQiOiAiVHJhaW5pbmcgdXNlZCBjb21wZXRpdGlvbiB0cmFpbmluZyBpbWFnZXMgYW5kIGFubm90YXRpb25zIG9ubHk7IG1vZGVsIHB1Ymxpc2hlciByZXBvcnRzIG5vIHB1YmxpYyBNQUdGaUxPL3Rlc3Qtb3ZlcmxhcCBsYWJlbHMuIiwKICAgICAgICAic291cmNlX21ldGhvZF9yZWZlcmVuY2UiOiAiaHR0cHM6Ly93d3cua2FnZ2xlLmNvbS9jb2RlL3JvYmluYW1pcmJhaGFyL3NvbGFyLWZpbGFtZW50LXBhbm9wdGljLW1vcnBob2xvZ3ktc2VnbWVudGF0aW9uIiwKICAgICAgICAiY29tcGV0aXRpb25fcm9vdCI6IHN0cihjb21wZXRpdGlvbl9yb290KSwKICAgICAgICAidHJhaW5pbmdfYW5ub3RhdGlvbnNfcmVhZCI6IEZhbHNlLAogICAgICAgICJ0ZXN0X2xhYmVsc19yZWFkIjogRmFsc2UsCiAgICAgICAgInRlc3RfaW1hZ2VzX3Byb2Nlc3NlZCI6IGxlbih0ZXN0X2ZpbGVzKSwKICAgICAgICAiZGV0ZWN0b3JfY2hlY2twb2ludCI6IHsKICAgICAgICAgICAgIm5hbWUiOiBkZXRlY3Rvcl9wYXRoLm5hbWUsCiAgICAgICAgICAgICJtZDUiOiBfZGlnZXN0KGRldGVjdG9yX3BhdGgsICJtZDUiKSwKICAgICAgICAgICAgInNoYTI1NiI6IF9kaWdlc3QoZGV0ZWN0b3JfcGF0aCwgInNoYTI1NiIpLAogICAgICAgICAgICAiYnl0ZXMiOiBkZXRlY3Rvcl9wYXRoLnN0YXQoKS5zdF9zaXplLAogICAgICAgIH0sCiAgICAgICAgInJlZmluZXJfY2hlY2twb2ludCI6IHsKICAgICAgICAgICAgIm5hbWUiOiByZWZpbmVyX3BhdGgubmFtZSwKICAgICAgICAgICAgIm1kNSI6IF9kaWdlc3QocmVmaW5lcl9wYXRoLCAibWQ1IiksCiAgICAgICAgICAgICJzaGEyNTYiOiBfZGlnZXN0KHJlZmluZXJfcGF0aCwgInNoYTI1NiIpLAogICAgICAgICAgICAiYnl0ZXMiOiByZWZpbmVyX3BhdGguc3RhdCgpLnN0X3NpemUsCiAgICAgICAgfSwKICAgICAgICAidGhyZXNob2xkcyI6IHsKICAgICAgICAgICAgImRldGVjdG9yX3Njb3JlX21pbiI6IDAuNzAsCiAgICAgICAgICAgICJkZXRlY3Rvcl9tYXNrX21pbiI6IDAuNTAsCiAgICAgICAgICAgICJyZWZpbmVyX3RocmVzaG9sZCI6IDAuNzAsCiAgICAgICAgICAgICJ0dGFfdHJhbnNmb3JtcyI6IFsiaWRlbnRpdHkiLCAiaG9yaXpvbnRhbCIsICJ2ZXJ0aWNhbCIsICJib3RoIl0sCiAgICAgICAgfSwKICAgICAgICAic3VibWlzc2lvbiI6IHsKICAgICAgICAgICAgKipjb3ZlcmFnZSwKICAgICAgICAgICAgInBhdGgiOiBzdHIoc3VibWlzc2lvbl9wYXRoKSwKICAgICAgICAgICAgInNoYTI1NiI6IF9kaWdlc3Qoc3VibWlzc2lvbl9wYXRoLCAic2hhMjU2IiksCiAgICAgICAgICAgICJieXRlcyI6IHN1Ym1pc3Npb25fcGF0aC5zdGF0KCkuc3Rfc2l6ZSwKICAgICAgICB9LAogICAgICAgICJkaWFnbm9zdGljcyI6IGRpYWdub3N0aWNfcm93cywKICAgICAgICAiZWxhcHNlZF9zZWNvbmRzIjogZWxhcHNlZCwKICAgICAgICAicGVha19ncHVfbWVtb3J5X2J5dGVzIjogaW50KHRvcmNoLmN1ZGEubWF4X21lbW9yeV9hbGxvY2F0ZWQoKSksCiAgICAgICAgImRldmljZSI6IHRvcmNoLmN1ZGEuZ2V0X2RldmljZV9uYW1lKDApLAogICAgICAgICJoYXJkd2FyZSI6IGhhcmR3YXJlX2luZm8oKSwKICAgICAgICAicHl0aG9uIjogcGxhdGZvcm0ucHl0aG9uX3ZlcnNpb24oKSwKICAgICAgICAidG9yY2giOiB0b3JjaC5fX3ZlcnNpb25fXywKICAgICAgICAidG9yY2h2aXNpb24iOiBfX2ltcG9ydF9fKCJ0b3JjaHZpc2lvbiIpLl9fdmVyc2lvbl9fLAogICAgICAgICJwYWNrYWdlX3ZlcnNpb25zIjogcGFja2FnZV92ZXJzaW9ucygpLAogICAgfQogICAgcmVwb3J0X3BhdGggPSBvdXRwdXRfZGlyIC8gInJ1bl9yZXBvcnQuanNvbiIKICAgIHJlcG9ydF9wYXRoLndyaXRlX3RleHQoanNvbi5kdW1wcyhyZXBvcnQsIGluZGVudD0yLCBzb3J0X2tleXM9VHJ1ZSkgKyAiXG4iLCBlbmNvZGluZz0idXRmLTgiKQogICAgcHJpbnQoanNvbi5kdW1wcyh7a2V5OiB2YWx1ZSBmb3Iga2V5LCB2YWx1ZSBpbiByZXBvcnQuaXRlbXMoKSBpZiBrZXkgIT0gImRpYWdub3N0aWNzIn0sIGluZGVudD0yKSwgZmx1c2g9VHJ1ZSkKICAgIHJldHVybiAwCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIHJhaXNlIFN5c3RlbUV4aXQobWFpbigpKQo="}
for relative, payload in SOURCE_FILES.items():
    destination = REPOSITORY_PATH / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(base64.b64decode(payload))
print('Source commit:', SOURCE_COMMIT)
print('Source bundle SHA-256:', SOURCE_BUNDLE_SHA256)


In [ ]:
import os, subprocess, sys
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['PYTHONPATH'] = str(REPOSITORY_PATH / 'src')
command = [sys.executable, '-u', str(REPOSITORY_PATH / 'scripts' / 'run_external_panoptic_inference.py'), '--data-root', '/kaggle/input', '--model-root', '/kaggle/input', '--output-dir', '/kaggle/working/equinexus/public-i079']
print('Running test-only inference:', ' '.join(command), flush=True)
subprocess.run(command, check=True)


In [ ]:
from pathlib import Path
run_root = Path('/kaggle/working/equinexus/public-i079')
for name in ('run_report.json', 'submission.csv'):
    path = run_root / name
    print(name, 'exists=', path.is_file(), 'bytes=', path.stat().st_size if path.is_file() else 0)
if (run_root / 'run_report.json').is_file():
    print((run_root / 'run_report.json').read_text(encoding='utf-8'))
